# Predicting Coral Bleaching Severity with Multilayer Perceptrons
### A Deep Dive into How Network Depth and Width Shape What a Neural Network Can Learn

**Author:** [Tahsan Mahmud]  
**Course:** [Machine Learning and Neural Networks]  
**GitHub:** [https://github.com/Tahsan-Md/Coral_bleaching_mlp_tutorial.git]

---

## Introduction

Coral reefs support roughly 25% of all marine species, yet they are under severe threat from ocean warming and acidification. One of the most visible symptoms is **coral bleaching** — a process where corals expel the algae living in their tissues, turning white and becoming vulnerable to death.

In this tutorial, we use a **Multilayer Perceptron (MLP)** — a foundational feedforward neural network — to attempt to predict bleaching severity (Low, Medium, High) from environmental measurements such as sea surface temperature (SST) and ocean pH.

Along the way, we explore a key architectural question:

> **How does the depth (number of layers) and width (neurons per layer) of an MLP affect its ability to learn — and what happens when the data itself has no clear signal?**

This tutorial teaches both how MLPs work *and* how to interpret their results critically.

### What you will learn
- What an MLP is and how forward/backward propagation works
- How to prepare tabular data for neural network training
- How to build and compare MLPs of varying depth and width
- How to evaluate models honestly — including recognising when a dataset lacks predictive signal
- Why data quality matters as much as model architecture

### References
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press. https://www.deeplearningbook.org/
- Scikit-learn MLPClassifier: https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html
- Hughes, T. P. et al. (2017). Global warming and recurrent mass bleaching of corals. *Nature*, 543, 373-377. https://doi.org/10.1038/nature21707
- Universal Approximation Theorem: https://en.wikipedia.org/wiki/Universal_approximation_theorem
- Cybenko, G. (1989). Approximation by superpositions of a sigmoidal function. *Mathematics of Control, Signals and Systems*, 2(4), 303-314.

---
## Part 1 — Background: What is a Multilayer Perceptron?

A **Multilayer Perceptron (MLP)** is the building block of modern deep learning. It consists of:

- An **input layer** — one node per input feature
- One or more **hidden layers** — where learning happens
- An **output layer** — one node per class (for classification)

```
Input Layer       Hidden Layer 1     Hidden Layer 2     Output Layer

SST ──────────►   [ neuron ]          [ neuron ]         [ Low   ]
pH  ──────────►   [ neuron ]  ──────► [ neuron ]  ─────► [ Medium]
Lat ──────────►   [ neuron ]          [ neuron ]         [ High  ]
Lon ──────────►   [ neuron ]
Species ──────►   [ neuron ]
Heatwave ─────►
```

### How does a neuron work?
Each neuron computes: **output = activation(W · x + b)**  
where W are learned weights, x is the input, b is a bias, and the activation adds non-linearity. We use **ReLU**: `f(x) = max(0, x)`.

### Training
The network learns by minimising **cross-entropy loss** using **backpropagation** and the **Adam optimiser**.

### Depth vs. Width
| | **Depth** | **Width** |
|---|---|---|
| Definition | Number of hidden layers | Neurons per hidden layer |
| Increases | Hierarchical abstraction | Capacity per layer |
| Risk | Vanishing gradients, overfitting | Overfitting, slow training |

The **Universal Approximation Theorem** (Cybenko, 1989) proves a single hidden layer can approximate any continuous function — but deeper networks often learn more efficiently with fewer parameters.

---
## Part 2 — Setup

In [ ]:
# Uncomment if needed:
# !pip install pandas numpy scikit-learn matplotlib seaborn scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.dummy import DummyClassifier
from scipy.stats import f_oneway

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries loaded.")

---
## Part 3 — Exploratory Data Analysis

In [ ]:
df = pd.read_csv('realistic_ocean_climate_dataset.csv')  # update path if needed
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nBleaching Severity distribution:")
print(df['Bleaching Severity'].value_counts())
print(f"\nUnique locations: {list(df['Location'].unique())}")

In [ ]:
# Figure 1: Class distribution & SST histogram by severity
df_clean = df.dropna(subset=['Bleaching Severity'])
colors = ['#2ecc71', '#f39c12', '#e74c3c']  # accessible: green, orange, red

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sc = df_clean['Bleaching Severity'].value_counts()[['Low', 'Medium', 'High']]
bars = axes[0].bar(sc.index, sc.values, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Bleaching Severity Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Severity'); axes[0].set_ylabel('Count')
for bar, count in zip(bars, sc.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height()+1,
                 str(count), ha='center', fontweight='bold')

for sev, color in zip(['Low', 'Medium', 'High'], colors):
    axes[1].hist(df_clean[df_clean['Bleaching Severity']==sev]['SST (\u00b0C)'],
                 bins=15, alpha=0.6, color=color, label=sev, edgecolor='black', linewidth=0.5)
axes[1].set_title('SST Distribution by Bleaching Severity', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sea Surface Temperature (\u00b0C)'); axes[1].set_ylabel('Count')
axes[1].legend(title='Severity')

plt.tight_layout()
plt.savefig('fig1_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: Feature correlation heatmap
fig, ax = plt.subplots(figsize=(8, 5))
numeric_cols = ['SST (\u00b0C)', 'pH Level', 'Latitude', 'Longitude', 'Species Observed']
sns.heatmap(df_clean[numeric_cols].corr(), annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: SST vs pH coloured by severity
fig, ax = plt.subplots(figsize=(8, 5))
color_map = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
for sev in ['Low', 'Medium', 'High']:
    s = df_clean[df_clean['Bleaching Severity']==sev]
    ax.scatter(s['SST (\u00b0C)'], s['pH Level'], c=color_map[sev], label=sev,
               alpha=0.6, s=40, edgecolors='black', linewidth=0.3)
ax.set_xlabel('Sea Surface Temperature (\u00b0C)', fontsize=11)
ax.set_ylabel('pH Level', fontsize=11)
ax.set_title('SST vs pH Level by Bleaching Severity', fontsize=13, fontweight='bold')
ax.legend(title='Bleaching Severity')
plt.tight_layout()
plt.savefig('fig3_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical signal check: ANOVA test
print("One-way ANOVA — do mean feature values differ significantly between severity classes?")
print("(p < 0.05 suggests a feature is useful for predicting severity)\n")

for feat in ['SST (\u00b0C)', 'pH Level', 'Species Observed']:
    groups = [df_clean[df_clean['Bleaching Severity']==sev][feat].values
              for sev in ['Low', 'Medium', 'High']]
    f_stat, p_val = f_oneway(*groups)
    flag = '\u26a0 NOT significant' if p_val > 0.05 else '\u2713 significant'
    print(f"{feat:>20}: F={f_stat:.3f}, p={p_val:.4f}  {flag}")

**Key insight:** All p-values are >> 0.05, meaning **no feature significantly separates the three bleaching classes**. This indicates the dataset is synthetic — bleaching labels were assigned independently of environmental features. Keep this in mind when interpreting model results below.

> **Takeaway:** Always check whether your features actually relate to your target before building a model. If they don't, no architecture — however deep or wide — can find signal that isn't there.

---
## Part 4 — Data Preprocessing

In [ ]:
# Drop rows with missing target
df_model = df.dropna(subset=['Bleaching Severity']).copy()
print(f"Usable rows: {len(df_model)} (dropped {len(df) - len(df_model)} with missing labels)")

# Encode target
label_encoder = LabelEncoder()
df_model['target'] = label_encoder.fit_transform(df_model['Bleaching Severity'])
print(f"Encoding: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

# Encode boolean feature
df_model['Heatwave'] = df_model['Marine Heatwave'].astype(int)

FEATURES = ['SST (\u00b0C)', 'pH Level', 'Latitude', 'Longitude', 'Species Observed', 'Heatwave']
X = df_model[FEATURES].values
y = df_model['target'].values

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Feature scaling — fit on train only!
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)   # transform only — never fit on test data
X_all_s = StandardScaler().fit_transform(X)  # for cross-validation

print(f"\nTrain: {X_train_s.shape}, Test: {X_test_s.shape}")
print("\nWhy scale? Gradient updates are proportional to input magnitude.")
print("Without scaling, Longitude (-155 to +148) would dominate pH (7.87 to 8.20).")

---
## Part 5 — Establishing a Baseline

In [ ]:
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(X_train_s, y_train)
baseline_acc = dummy.score(X_test_s, y_test)

print(f"Baseline accuracy (always predict most frequent class): {baseline_acc:.3f}")
print(f"Class 'Low': {(y==0).sum()}/{len(y)} samples = {(y==0).mean():.1%}")
print(f"\nAny useful model must clearly beat {baseline_acc:.1%}.")

---
## Part 6 — Experiment 1: Effect of Network Depth

We fix **width = 64 neurons** and vary depth from 1 to 5 hidden layers.  
We use **5-fold cross-validation** for reliable estimates with small data, and report mean ± std.

In [ ]:
WIDTH = 64
depths = [1, 2, 3, 4, 5]
d_means, d_stds = [], []

print(f"{'Depth':>6}  {'Architecture':>15}  {'CV Mean':>8}  {'CV Std':>7}")
print("-" * 42)

for d in depths:
    arch = tuple([WIDTH] * d)
    scores = cross_val_score(
        MLPClassifier(hidden_layer_sizes=arch, activation='relu',
                      solver='adam', max_iter=1000, random_state=RANDOM_STATE),
        X_all_s, y, cv=5, scoring='accuracy'
    )
    d_means.append(scores.mean()); d_stds.append(scores.std())
    print(f"{d:>6}  {str(arch):>15}  {scores.mean():>8.3f}  {scores.std():>7.3f}")

print(f"\nBaseline: {baseline_acc:.3f}")

In [ ]:
# Figure 4: Depth effect with error bars
fig, ax = plt.subplots(figsize=(8, 5))

ax.errorbar(depths, d_means, yerr=d_stds, fmt='o-', color='#2980b9',
            linewidth=2, markersize=8, capsize=5, label='CV Accuracy (\u00b11 std)')
ax.axhline(baseline_acc, color='#7f8c8d', linestyle='--',
           linewidth=1.5, label=f'Baseline ({baseline_acc:.2f})')

ax.set_xlabel('Number of Hidden Layers (Depth)', fontsize=12)
ax.set_ylabel('5-Fold CV Accuracy', fontsize=12)
ax.set_title('Effect of Network Depth on MLP Accuracy\n(Width = 64 neurons per layer)',
             fontsize=12, fontweight='bold')
ax.set_xticks(depths); ax.set_ylim(0.1, 0.65)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

for d, m in zip(depths, d_means):
    ax.annotate(f'{m:.3f}', xy=(d, m), xytext=(5, 10),
                textcoords='offset points', fontsize=9, color='#2980b9')

plt.tight_layout()
plt.savefig('fig4_depth_effect.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 7 — Experiment 2: Effect of Network Width

We fix **depth = 2 layers** and vary width from 8 to 256 neurons per layer.

In [ ]:
DEPTH = 2
widths = [8, 16, 32, 64, 128, 256]
w_means, w_stds = [], []

print(f"{'Width':>6}  {'Architecture':>12}  {'CV Mean':>8}  {'CV Std':>7}")
print("-" * 40)

for w in widths:
    arch = tuple([w] * DEPTH)
    scores = cross_val_score(
        MLPClassifier(hidden_layer_sizes=arch, activation='relu',
                      solver='adam', max_iter=1000, random_state=RANDOM_STATE),
        X_all_s, y, cv=5, scoring='accuracy'
    )
    w_means.append(scores.mean()); w_stds.append(scores.std())
    print(f"{w:>6}  {str(arch):>12}  {scores.mean():>8.3f}  {scores.std():>7.3f}")

print(f"\nBaseline: {baseline_acc:.3f}")

In [ ]:
# Figure 5: Width effect with error bars
fig, ax = plt.subplots(figsize=(8, 5))

ax.errorbar(range(len(widths)), w_means, yerr=w_stds, fmt='s--', color='#c0392b',
            linewidth=2, markersize=8, capsize=5, label='CV Accuracy (\u00b11 std)')
ax.axhline(baseline_acc, color='#7f8c8d', linestyle='--',
           linewidth=1.5, label=f'Baseline ({baseline_acc:.2f})')

ax.set_xticks(range(len(widths))); ax.set_xticklabels([str(w) for w in widths])
ax.set_xlabel('Neurons per Layer (Width)', fontsize=12)
ax.set_ylabel('5-Fold CV Accuracy', fontsize=12)
ax.set_title('Effect of Network Width on MLP Accuracy\n(Depth = 2 hidden layers)',
             fontsize=12, fontweight='bold')
ax.set_ylim(0.1, 0.65); ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

for i, m in enumerate(w_means):
    ax.annotate(f'{m:.3f}', xy=(i, m), xytext=(5, 10),
                textcoords='offset points', fontsize=9, color='#c0392b')

plt.tight_layout()
plt.savefig('fig5_width_effect.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 8 — Side-by-Side Comparison

In [ ]:
# Figure 6: Combined depth vs width
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, xs, means, stds, fmt, color, xlabel, title in [
    (axes[0], depths, d_means, d_stds, 'o-', '#2980b9',
     'Number of Hidden Layers', 'Effect of Depth\n(Width = 64)'),
    (axes[1], range(len(widths)), w_means, w_stds, 's--', '#c0392b',
     'Neurons per Layer', 'Effect of Width\n(Depth = 2)'),
]:
    ax.errorbar(xs, means, yerr=stds, fmt=fmt, color=color,
                linewidth=2, markersize=8, capsize=5, label='CV Accuracy (\u00b11 std)')
    ax.axhline(baseline_acc, color='#7f8c8d', linestyle='--',
               linewidth=1.5, label=f'Baseline ({baseline_acc:.2f})')
    ax.set_xlabel(xlabel, fontsize=11); ax.set_ylabel('5-Fold CV Accuracy', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(0.1, 0.65); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

axes[1].set_xticks(range(len(widths))); axes[1].set_xticklabels([str(w) for w in widths])

plt.suptitle('MLP Depth vs. Width — Coral Bleaching Severity Classification',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig6_depth_vs_width.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 9 — Best Model: Detailed Evaluation

In [ ]:
# Train best architecture on full train split
best_arch = (64,)  # 1 hidden layer of 64 — most stable across experiments

best_mlp = MLPClassifier(
    hidden_layer_sizes=best_arch,
    activation='relu', solver='adam',
    max_iter=1000, random_state=RANDOM_STATE
)
best_mlp.fit(X_train_s, y_train)
y_pred = best_mlp.predict(X_test_s)

print(f"Test accuracy: {best_mlp.score(X_test_s, y_test):.3f}")
print(f"Baseline:      {baseline_acc:.3f}\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
# Figure 7: Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_).plot(
    ax=ax, cmap='Blues', colorbar=False
)
ax.set_title(f'Confusion Matrix \u2014 MLP {best_arch}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig7_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 8: Training loss curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(best_mlp.loss_curve_, color='#2980b9', linewidth=2)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Training Loss (Cross-Entropy)', fontsize=11)
ax.set_title(f'Training Loss Curve \u2014 MLP {best_arch}', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig8_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 10 — Interpreting the Results

### What did we find?

Across all depth and width configurations, CV accuracy remained around **32–37%** — at or below the baseline of ~37.7%. This is not a failure of the MLP architecture. It is a direct consequence of the dataset: the ANOVA test showed that bleaching labels are **statistically independent** of all measured features.

> **No model — however deep or wide — can extract signal that does not exist in the data.**

### What this teaches us

| Lesson | What we observed |
|--------|------------------|
| **Check your data first** | ANOVA p-values > 0.85 — clear red flag before any modelling |
| **Depth cannot compensate for noise** | All depths performed similarly — no hierarchical structure to learn |
| **Width has diminishing returns** | Wider models showed no consistent improvement |
| **Cross-validation reveals instability** | Large std (\u00b10.04-0.06) shows high variance from noise |
| **Always compare to baseline** | Without a baseline, ~35% accuracy might look acceptable |

### What would happen with real data?

Hughes et al. (2017) show that mass bleaching is strongly correlated with thermal anomalies (degree heating weeks). A dataset with genuine SST-bleaching relationships would yield accuracy well above baseline, and we would observe a clear depth/width optimum demonstrating the **bias-variance tradeoff**:
- Too shallow/narrow → underfitting (high bias)
- Too deep/wide → overfitting (high variance)

---
## Part 11 — Summary, Ethics, and Further Reading

### Key Takeaways
1. **Feature scaling is non-negotiable** — always normalise before training an MLP
2. **Establish a baseline first** — a model barely beating random guessing has not learned
3. **Cross-validation > single split** — especially on small datasets
4. **Data quality > architecture** — no amount of depth or width rescues a bad dataset
5. **The Universal Approximation Theorem has a catch** — it assumes the relationship exists in the data

### Ethical Considerations
- **False negatives** (predicting Low when severity is High) could delay conservation responses
- Models trained on synthetic or limited data must never be deployed without validation on real field data
- Geographic bias — 7 reef systems cannot generalise globally
- Data collection should follow ethical marine research protocols

### Further Reading
- Goodfellow et al. (2016). *Deep Learning*. https://www.deeplearningbook.org/
- Hughes et al. (2017). *Nature*. https://doi.org/10.1038/nature21707
- Cybenko (1989). Approximation by superpositions of a sigmoidal function. *MCSS*.
- Scikit-learn MLPClassifier: https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html

In [ ]:
# Final results summary
print("=" * 55)
print("FINAL RESULTS SUMMARY")
print("=" * 55)
print(f"Dataset: {len(df_model)} labelled samples, 6 features, 3 classes")
print(f"Baseline: {baseline_acc:.3f}\n")

print("DEPTH EXPERIMENT (CV=5, width=64):")
for d, m, s in zip(depths, d_means, d_stds):
    print(f"  Depth {d}: {m:.3f} \u00b1 {s:.3f}")

print("\nWIDTH EXPERIMENT (CV=5, depth=2):")
for w, m, s in zip(widths, w_means, w_stds):
    print(f"  Width {w:>4}: {m:.3f} \u00b1 {s:.3f}")

print(f"\nBest model test accuracy: {best_mlp.score(X_test_s, y_test):.3f}")
print("=" * 55)